# EINet Sweep — Results Summary

Reads all `performance/final.json` files dynamically. No hardcoded accuracies.

**Effective params** = active synaptic connections in the forward pass (TopKLinear K-sparse, not stored dense matrix).
Formula: per EINet layer, `N_E × n_branches × (ee + ie) + N_I × n_branches × ei`  
where `n_branches=6` (2×2) or `39` (3×3×3), `ee=40, ie=10, ei=10`, `N_I=N_E//4`.

This is ~7% below the empirical counts in the briefing (small bias/aggregation weights not counted here).
Point neuron effective = `N_E × 4096` (dense linear, all connections active).

In [ ]:
import json, re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from collections import defaultdict

OUTPUTS = Path("/n/holylfs06/LABS/kempner_project_b/Lab/dendritic/adivm/dendritic-modeling/outputs")

# ── architecture constants (fixed across every sweep) ───────────────────────
EE, IE, EI   = 40, 10, 10
E_I_RATIO    = 4
N_BRANCHES   = {"2x2": 6, "3x3x3": 39}   # active branch levels (soma excluded)

def eff_einet(ne, morph, n_layers=1):
    """Active connections across n EINet layers.
    For n_layers=2 (k=2), second layer's E→E is clamped to min(EE, ne) because
    input_dim = ne which may be < 40 for small ne."""
    ni  = ne // E_I_RATIO
    nb  = N_BRANCHES[morph]
    # First layer: input from high-dim features, K-sparse so ee/ie not clamped
    first = ne * nb * (EE + IE) + ni * nb * EI
    if n_layers == 1:
        return first
    # Second layer: input from first layer (ne E + ni I outputs)
    ee2 = min(EE, ne)
    ie2 = min(IE, ni)
    ei2 = min(EI, ne)
    second = ne * nb * (ee2 + ie2) + ni * nb * ei2
    return first + second

def eff_point(ne, input_dim=4096):
    return ne * input_dim

# ── series definitions (order = color order; more-specific regex first) ─────
# Validated categorical palette from dataviz skill (light mode, CVD-safe order)
SERIES = [
    dict(regex=r"^alexnet_k1_ne(\d+)_noproj_ddp",            label="fc7 only, 2×2",      color="#2a78d6", ls="-",  eff=lambda ne: eff_einet(ne, "2x2")),
    dict(regex=r"^alexnet_k1_ne(\d+)_3x3x3_noproj_ddp",      label="fc7 only, 3×3×3",    color="#1baf7a", ls="-",  eff=lambda ne: eff_einet(ne, "3x3x3")),
    dict(regex=r"^alexnet_k1_ne(\d+)_point_noproj_ddp",       label="fc7 only, point",    color="#eda100", ls="-",  eff=lambda ne: eff_point(ne)),
    dict(regex=r"^alexnet_fc6fc7_ne(\d+)_2x2_ddp",            label="fc6+fc7 k=1, 2×2",   color="#008300", ls="-",  eff=lambda ne: eff_einet(ne, "2x2")),
    dict(regex=r"^alexnet_fc6fc7_ne(\d+)_3x3x3_ddp",          label="fc6+fc7 k=1, 3×3×3", color="#4a3aa7", ls="-",  eff=lambda ne: eff_einet(ne, "3x3x3")),
    dict(regex=r"^alexnet_fc6fc7_k2_ne(\d+)_2x2_ddp",         label="fc6+fc7 k=2, 2×2",   color="#e34948", ls="--", eff=lambda ne: eff_einet(ne, "2x2",   n_layers=2)),
    dict(regex=r"^alexnet_fc6fc7_k2_ne(\d+)_3x3x3_ddp",       label="fc6+fc7 k=2, 3×3×3", color="#e87ba4", ls="--", eff=lambda ne: eff_einet(ne, "3x3x3", n_layers=2)),
]

print("Series defined:", [s['label'] for s in SERIES])

: 

In [ ]:
# ── load all final.json results ──────────────────────────────────────────────
def read_test_acc(run_dir):
    fj = run_dir / "performance" / "final.json"
    if fj.exists():
        d = json.load(open(fj))
        return d.get("accuracy", {}).get("test")
    return None

# Load baselines (different file format)
baseline_dirs = ["alexnet_baseline", "alexnet_baseline_seed41",
                 "alexnet_baseline_seed42", "alexnet_baseline_seed43"]
baseline_accs = []
for bd in baseline_dirs:
    fp = OUTPUTS / bd / "performance.json"
    if fp.exists():
        d = json.load(open(fp))
        baseline_accs.append(d["test"]["top1_accuracy"])
BASELINE_FROZEN = np.mean(baseline_accs) if baseline_accs else 0.5257  # frozen AlexNet fine-tune fc8

pretrained_fc7_dir = OUTPUTS / "alexnet_k1_fc7_pretrained_finetune_fc8_ddp_20260421_213838"
pretrained_fc7_acc = read_test_acc(pretrained_fc7_dir)

print(f"Baseline (frozen features + fc6, fine-tune fc8):  {BASELINE_FROZEN:.4f}")
print(f"Baseline (pretrained fc7 kept, fine-tune fc8):   {pretrained_fc7_acc:.4f}")

# Classify and collect all sweep results
# data[label] = dict of ne -> list of accs (take max to deduplicate reruns)
data = defaultdict(lambda: defaultdict(list))

for run_dir in sorted(OUTPUTS.iterdir()):
    acc = read_test_acc(run_dir)
    if acc is None:
        continue
    name = run_dir.name
    for s in SERIES:
        m = re.match(s["regex"], name)
        if m:
            ne = int(m.group(1))
            data[s["label"]][ne].append(acc)
            break  # first match wins

# Summarise: take max across any duplicate runs at same ne
sweep = {}  # label -> sorted (ne, acc, eff_params)
for s in SERIES:
    label = s["label"]
    if label not in data:
        sweep[label] = []
        continue
    rows = []
    for ne, accs in sorted(data[label].items()):
        best_acc = max(accs)
        rows.append((ne, best_acc * 100, s["eff"](ne)))
    sweep[label] = rows
    print(f"{label}: {[r[0] for r in rows]}")

In [ ]:
# ── Plot 1: Accuracy vs N_E ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5.5))

for s in SERIES:
    rows = sweep[s["label"]]
    if not rows:
        continue
    nes, accs, _ = zip(*rows)
    marker = "o" if s["ls"] == "-" else "s"
    zorder = 3 if s["ls"] == "-" else 2
    ax.plot(nes, accs, marker=marker, color=s["color"], ls=s["ls"],
            lw=2, ms=7, label=s["label"], zorder=zorder,
            markeredgewidth=0.5, markeredgecolor="white")

# Baselines
ax.axhline(BASELINE_FROZEN * 100, color="#52514e", ls=":", lw=1.5,
           label=f"Frozen AlexNet + fc8  ({BASELINE_FROZEN*100:.1f}%)")
if pretrained_fc7_acc:
    ax.axhline(pretrained_fc7_acc * 100, color="#0b0b0b", ls=":", lw=1.5,
               label=f"Pretrained fc7 + fc8  ({pretrained_fc7_acc*100:.1f}%)")

ax.set_xscale("log", base=2)
ne_ticks = [32, 64, 128, 256, 512, 1024, 2048, 4096]
ax.set_xticks(ne_ticks)
ax.set_xticklabels(ne_ticks, fontsize=9)
ax.set_xlabel("N_E (excitatory neurons)", fontsize=11)
ax.set_ylabel("Top-1 Test Accuracy (ImageNet)", fontsize=11)
ax.set_title("EINet Sweeps — Accuracy vs Neuron Count", fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.grid(True, alpha=0.25, lw=0.8)
ax.spines[["top", "right"]].set_visible(False)

legend = ax.legend(fontsize=8.5, framealpha=0.9, edgecolor="#d5d4cf",
                   loc="upper left", bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.savefig("acc_vs_ne.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 2: Accuracy vs Effective Parameters ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5.5))

for s in SERIES:
    rows = sweep[s["label"]]
    if not rows:
        continue
    nes, accs, effs = zip(*rows)
    effs_M = [e / 1e6 for e in effs]
    marker = "o" if s["ls"] == "-" else "s"
    zorder = 3 if s["ls"] == "-" else 2
    ax.plot(effs_M, accs, marker=marker, color=s["color"], ls=s["ls"],
            lw=2, ms=7, label=s["label"], zorder=zorder,
            markeredgewidth=0.5, markeredgecolor="white")

ax.axhline(BASELINE_FROZEN * 100, color="#52514e", ls=":", lw=1.5,
           label=f"Frozen AlexNet + fc8  ({BASELINE_FROZEN*100:.1f}%)")
if pretrained_fc7_acc:
    ax.axhline(pretrained_fc7_acc * 100, color="#0b0b0b", ls=":", lw=1.5,
               label=f"Pretrained fc7 + fc8  ({pretrained_fc7_acc*100:.1f}%)")

# Vertical references: dense FC layer param counts
fc7_params_M    = (4096 * 4096 + 4096) / 1e6  # 16.78M
fc6_params_M    = (9216 * 4096 + 4096) / 1e6  # 37.75M
fc6fc7_params_M = fc6_params_M + fc7_params_M  # 54.53M

for x, label in [
    (fc7_params_M,    "fc7\n(16.8M)"),
    (fc6_params_M,    "fc6\n(37.8M)"),
    (fc6fc7_params_M, "fc6+fc7\n(54.5M)"),
]:
    ax.axvline(x, color="#52514e", ls="-.", lw=1, alpha=0.5)
    ax.annotate(label, xy=(x, 0.04), xycoords=("data", "axes fraction"),
                xytext=(3, 0), textcoords="offset points",
                fontsize=7, color="#52514e", va="bottom")

ax.set_xscale("log")
ax.set_xlabel("Effective Parameters (M active connections, log scale)", fontsize=11)
ax.set_ylabel("Top-1 Test Accuracy (ImageNet)", fontsize=11)
ax.set_title("EINet Sweeps — Accuracy vs Effective Parameters", fontsize=12, fontweight="bold")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.3g}M" if x < 1 else f"{x:.1f}M"))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.grid(True, alpha=0.25, lw=0.8)
ax.spines[["top", "right"]].set_visible(False)

# Legend below the plot, 3 columns so it stays compact
ax.legend(fontsize=8.5, framealpha=0.9, edgecolor="#d5d4cf",
          loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)

plt.tight_layout()
plt.savefig("acc_vs_eff_params.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
print(f"{'Experiment':<30} {'N_E':>6} {'Test Acc':>9} {'Eff params':>12}")
print("-" * 62)
for s in SERIES:
    rows = sweep[s["label"]]
    if not rows:
        continue
    print()
    for ne, acc, eff in rows:
        status = "" if len(data[s['label']][ne]) == 1 else f" ({len(data[s['label']][ne])} runs, best)"
        print(f"{s['label']:<30} {ne:>6} {acc:>8.2f}%  {eff/1e6:>9.3f}M{status}")

print()
print("-" * 62)
print(f"{'Frozen AlexNet + fc8 (baseline)':<30}        {BASELINE_FROZEN*100:>8.2f}%")
if pretrained_fc7_acc:
    print(f"{'Pretrained fc7 + fc8 (baseline)':<30}        {pretrained_fc7_acc*100:>8.2f}%")